In [1]:
import pandas as pd

def run_check():
    """Run the check without refreshing Excel connections"""
    
    # Load files
    valid_versions = pd.read_excel(r"C:\Users\falve11887\OneDrive - Elekta\Bartalo, Renan's files - Project - Valid Version\Versions_Valid.xlsx")
    donloaded_file = r'C:\Users\falve11887\Downloads\all sw.csv'
    
    clm_data = pd.read_csv(donloaded_file)
    
    product_aliases = {
        "srp - monaco software": "monaco software"
    }

    def normalize_name(name):
        name_clean = str(name).strip().lower()
        return product_aliases.get(name_clean, name_clean)

    valid_set = set(
        f"{normalize_name(row['Product Sibex Name'])}|{str(row['Version Detail']).strip().lower()}"
        for _, row in valid_versions.iterrows()
    )

    invalid_results = []
    
    for _, row in clm_data.iterrows():
        product_name = str(row['Installed Product: Installed Product']).strip()
        sibex_name = str(row['Sibex Name']).strip()
        version = str(row['Version Detail']).strip()
        id = row.get('Installed Product: ID', '') 
        warranty_date = row.get('Warranty Start Date', '') 
        order_line = row.get('Order Line Item', '')
        
        # Check if version is empty
        if version == '' or version.lower() == 'nan':
            invalid_results.append({
                'Warranty Start Date': warranty_date, 
                'Order Line Item': order_line,             
                'Installed Product: Installed Product': product_name,
                'Installed Product: ID': id,
                'Sibex Name': sibex_name,
                'Version Detail': version,
                'Reason why is in the spreadsheet': 'Empty version in CLM'
            })
            continue
            
        normalized_sibex = normalize_name(sibex_name)
        key = f"{normalized_sibex}|{version.lower()}"
        
        if key not in valid_set:
            invalid_results.append({
                'Warranty Start Date': warranty_date, 
                'Order Line Item': order_line,            
                'Installed Product: Installed Product': product_name,
                'Installed Product: ID': id,
                'Sibex Name': sibex_name,
                'Version Detail': version,
                'Reason why is in the spreadsheet': 'Version not in valid list'
            })
    
    if invalid_results:
        result_df = pd.DataFrame(invalid_results)
        result_df['Warranty Start Date'] = pd.to_datetime(result_df['Warranty Start Date'], errors='coerce')
        
        # ----------------------------------------------------------------------
        # DYNAMIC DATE RANGE CALCULATION:
        # Excludes current month & last month (e.g., if August -> excludes Aug & Jul, keeps up to Jun)
        # ----------------------------------------------------------------------
        today = pd.Timestamp.now()
        
        # First day of the current month
        first_day_this_month = today.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
        
        # Upper limit: strictly before the 1st of last month
        upper_limit = first_day_this_month - pd.DateOffset(months=1)
        
        # Lower limit: 24 months prior to the upper limit
        cutoff_date = upper_limit - pd.DateOffset(months=24)
        
        # FILTER: Keep dates between cutoff_date (inclusive) and upper_limit (exclusive)
        result_df = result_df[
            (result_df['Warranty Start Date'] >= cutoff_date) & 
            (result_df['Warranty Start Date'] < upper_limit)
        ]
        
    else:
        result_df = pd.DataFrame(columns=[
            'Warranty Start Date',
            'Order Line Item',
            'Installed Product: Installed Product',
            'Installed Product: ID',
            'Sibex Name', 
            'Version Detail',
            'Reason why is in the spreadsheet'
        ])
    
    # Sort remaining results ascending by Warranty Start Date
    result_df = result_df.sort_values(by='Warranty Start Date', ascending=True)
    
    return result_df

# Run the check
invalid_products = run_check()
print(f"Found {len(invalid_products)} invalid products")

# Show results
if not invalid_products.empty:
    print("\nInvalid Products:")
    print(invalid_products.head())
    
    # Save to Excel
    output_file = 'manual_invalid_versions_recent_installations.xlsx'
    invalid_products.to_excel(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
else:
    print("\nAll products have valid versions!")

Found 274 invalid products

Invalid Products:
    Warranty Start Date Order Line Item  \
375          2024-07-10             NaN   
376          2024-07-15             NaN   
374          2024-07-15             NaN   
378          2024-07-23             NaN   
377          2024-07-29             NaN   

           Installed Product: Installed Product Installed Product: ID  \
375             MOA Software/10587-MOA001/5.0.1       a0R6g00000HfEqf   
376            MOA Software/11320-MOA001/4.20.5       a0R6g00000HfH2a   
374            MOA Software/10024-MOA001/4.20.5       a0R6g00000HfH2c   
378  MOSAIQ VOICE SOFTWARE/13708-PAL001/2.2.2.9       a0R6g00000HfJUv   
377    MOA Software/12629-MOA001/Non-Serialized       a0R6g00000HfLOI   

                Sibex Name Version Detail Reason why is in the spreadsheet  
375           MOA Software          5.0.1        Version not in valid list  
376           MOA Software         4.20.5        Version not in valid list  
374           MOA Softwar